## ODS competition autumn 2025

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv('train_final.csv', index_col=0)

In [5]:
df.head()

,user_id,ts,gate_id
0,18,2022-07-29 09:08:54,7
1,18,2022-07-29 09:09:54,9
2,18,2022-07-29 09:09:54,9
3,18,2022-07-29 09:10:06,5
4,18,2022-07-29 09:10:08,5


In [6]:
df_final = pd.read_csv('test_final.csv', index_col=0)
df_final.head()

,ts,gate_id,user_word
37518,2023-01-03 08:21:00,9,gini
37519,2023-01-03 08:21:00,9,gini
37520,2023-01-03 08:21:18,5,gini
37521,2023-01-03 08:21:19,5,gini
37522,2023-01-03 08:21:39,10,gini


In [7]:
# --- 1. Feature Engineering (Полный набор данных) ---
df['ts'] = pd.to_datetime(df['ts'])
df = df.sort_values('ts')

# Создание базовых временных признаков
df['hour'] = df['ts'].dt.hour
df['min'] = df['ts'].dt.minute
df['day_of_week'] = df['ts'].dt.dayofweek
df['day_of_month'] = df['ts'].dt.day

# Cyclical Features (для Hour и DOW)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Признак взаимодействия (Gate + DOW)
df['gate_dow'] = df['gate_id'].astype(str) + '_' + df['day_of_week'].astype(str)

# Lag Feature (Числовой)
df['time_diff'] = df['ts'].diff().dt.total_seconds().fillna(0)
# Очистка
df = df.drop(columns=['ts'])

# --- 2. Target Encoding ---
# Кодируем user_id в числа (требуется для scikit-learn)
le = LabelEncoder()
df['user_id_encoded'] = le.fit_transform(df['user_id'])
Y = df['user_id_encoded']

# --- 3. Modeling Setup ---
X = df.drop(columns=['user_id', 'user_id_encoded'])


In [8]:
# Разделение данных (используем весь набор, без фильтрации)
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# --- 4. Preprocessing Pipeline (ColumnTransformer) ---
numerical_features = ['time_diff', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'min', 'day_of_month']
categorical_features = ['gate_id', 'day_of_week', 'hour', 'gate_dow']

# Создание преобразователя
preprocessor = ColumnTransformer(
    transformers=[
        # 1. Standard Scaler для числовых признаков (для регуляризации)
        ('num', StandardScaler(), numerical_features),
        # 2. OneHotEncoder для категориальных признаков
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features)
    ],
    remainder='passthrough' # Оставить остальные признаки как есть (на данный момент их нет)
)

# Преобразование данных
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

In [9]:
model=KNeighborsClassifier(n_neighbors=3, weights='distance',p=1 ,n_jobs=-1)
model.fit(X_train_processed, Y_train)

,n_neighbors,3
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,1
,metric,'minkowski'
,metric_params,None
,n_jobs,-1


In [10]:
# Предсказание на валидационном наборе
Y_preds = model.predict(X_val_processed)
acc = accuracy_score(Y_val, Y_preds)

# Декодирование меток обратно в исходные ID пользователей
Y_val_original = le.inverse_transform(Y_val)
Y_preds_original = le.inverse_transform(Y_preds)

print(f"\nValidation Accuracy (Model): {acc:.4f}")
print("Classification Report:")
print(classification_report(Y_val_original, Y_preds_original, zero_division=0))


Validation Accuracy (Model): 0.5167
Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.44      0.46       250
           1       0.55      0.50      0.52       260
           2       0.50      0.62      0.56         8
           3       0.51      0.57      0.54       198
           5       0.00      0.00      0.00         2
           6       0.53      0.57      0.55       403
           7       0.29      0.20      0.24        10
           8       1.00      0.33      0.50         6
           9       0.48      0.42      0.45       207
          10       0.50      0.33      0.40         3
          11       0.42      0.44      0.43       256
          12       0.53      0.58      0.55       391
          14       0.59      0.59      0.59       139
          15       0.49      0.50      0.49       351
          17       0.48      0.51      0.49       135
          18       0.62      0.62      0.62       316
          19       0.

In [11]:
# --- 1. Reusable Feature Engineering Function ---
def apply_features(df):
    df = df.copy()
    
    # Ensure 'ts' is the timestamp column
    ts_col_name = [c for c in df.columns if 'ts' in c.lower() or 'time' in c.lower()][0]
    df['ts'] = pd.to_datetime(df[ts_col_name])
    df = df.sort_values('ts')

    df['hour'] = df['ts'].dt.hour
    df['min'] = df['ts'].dt.minute
    df['day_of_week'] = df['ts'].dt.dayofweek
    df['day_of_month'] = df['ts'].dt.day

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    df['gate_dow'] = df['gate_id'].astype(str) + '_' + df['day_of_week'].astype(str)

    df['time_diff'] = df['ts'].diff().dt.total_seconds().fillna(0)
    
    df = df.drop(columns=['ts'])
    
    return df

In [ ]:
# Загружаем второй файл с буквенными обозначениями
df_letters = pd.read_csv('test_final.csv')

# Сохраняем буквенные обозначения для последующего сопоставления
letter_ids = df_letters['user_word'].copy()

# Применяем те же преобразования признаков, что и для тренировочных данных
df_letters['ts'] = pd.to_datetime(df_letters['ts'])
df_letters = df_letters.sort_values('ts')

df_letters['hour'] = df_letters['ts'].dt.hour
df_letters['min'] = df_letters['ts'].dt.minute
df_letters['day_of_week'] = df_letters['ts'].dt.dayofweek
df_letters['day_of_month'] = df_letters['ts'].dt.day

df_letters['hour_sin'] = np.sin(2 * np.pi * df_letters['hour'] / 24)
df_letters['hour_cos'] = np.cos(2 * np.pi * df_letters['hour'] / 24)
df_letters['dow_sin'] = np.sin(2 * np.pi * df_letters['day_of_week'] / 7)
df_letters['dow_cos'] = np.cos(2 * np.pi * df_letters['day_of_week'] / 7)

df_letters['gate_dow'] = df_letters['gate_id'].astype(str) + '_' + df_letters['day_of_week'].astype(str)
df_letters['time_diff'] = df_letters['ts'].diff().dt.total_seconds().fillna(0)
df_letters = df_letters.drop(columns=['ts'])

# Удаляем буквенный id из признаков для предсказания
X_letters = df_letters.drop(columns=['user_word'])

# Преобразуем признаки через обученный preprocessor
X_letters_processed = preprocessor.transform(X_letters)

# Делаем предсказания
predicted_encoded = model.predict(X_letters_processed)

# Декодируем предсказания в исходные id
predicted_ids = le.inverse_transform(predicted_encoded)

# Создаем таблицу соответствия
mapping_df = pd.DataFrame({
    'user_word': letter_ids,
    'preds': predicted_ids
})

# Для случаев, когда одному letter_id соответствует несколько predicted_id, 
# выбираем наиболее частый predicted_id для каждого letter_id
final_mapping = mapping_df.groupby('user_word')['preds'] \
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]) \
    .reset_index()

# Сохраняем результат
final_mapping.to_csv('answer_final.csv', index=False)

In [ ]:
#from sklearn.impute import SimpleImputer  imp = SimpleImputer(strategy='mean')  df[num_cols] = imp.fit_transform(df[num_cols])

In [ ]:
#sns.kdeplot(df[df[pred].isna()][target], label='NaN') sns.kdeplot(df[df[pred].notna()][target], label='No NaN')
# MCAR if different → no MCAR.
# How to convert predictions back from differences?
#y_restored = y_train_last + np.cumsum(y_diff_pred)